# Notebook 1 — Exploratory Data Analysis
Understanding the dataset before any modeling.
Key focus: class imbalance, feature distributions, why accuracy is meaningless.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.append(os.path.join('..', 'src'))
from fraud_detector import generate_synthetic_fraud_data

# Load data — swap path for real Kaggle CSV
df = generate_synthetic_fraud_data(n_samples=50000)
# df = pd.read_csv('../data/creditcard.csv')   # <- real data
print(df.shape)
df.head()

## 1. Class Distribution — The Core Problem

In [ ]:
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Class'].value_counts()
bars = axes[0].bar(['Legitimate', 'Fraud'], counts.values,
                   color=['#378ADD', '#E24B4A'], width=0.4)
axes[0].set_title('Transaction counts', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate\n99.83%', 'Fraud\n0.17%'],
            colors=['#378ADD', '#E24B4A'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class proportion', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {len(legit)/len(fraud):.0f}:1')

## 2. Why Accuracy is Meaningless

In [ ]:
naive_accuracy = len(legit) / len(df) * 100
print('=' * 50)
print('NAIVE MODEL: predict everything as legitimate')
print('=' * 50)
print(f'Accuracy:       {naive_accuracy:.2f}%  <- looks great!')
print(f'Frauds caught:  0 of {len(fraud)}  <- completely useless')
print(f'Recall:         0.00')
print(f'PR-AUC:         {len(fraud)/len(df):.4f}  <- baseline to beat')
print()
print('Lesson: NEVER use accuracy on imbalanced datasets.')
print('Use Precision, Recall, F1, and PR-AUC instead.')

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw amount
axes[0].hist(legit['Amount'].clip(0, 1000), bins=60, alpha=0.7,
             color='#378ADD', label='Legitimate', density=True)
axes[0].hist(fraud['Amount'].clip(0, 1000), bins=60, alpha=0.7,
             color='#E24B4A', label='Fraud', density=True)
axes[0].set_title('Transaction amount (clipped at $1000)')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Log amount
axes[1].hist(np.log1p(legit['Amount']), bins=60, alpha=0.7,
             color='#378ADD', label='Legitimate', density=True)
axes[1].hist(np.log1p(fraud['Amount']), bins=60, alpha=0.7,
             color='#E24B4A', label='Fraud', density=True)
axes[1].set_title('log(1 + Amount)  — after transformation')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg fraud amount:  ${fraud["Amount"].mean():.2f}')
print(f'Avg legit amount:  ${legit["Amount"].mean():.2f}')

## 4. PCA Feature Distributions — Fraud vs Legitimate

In [ ]:
# Top fraud-signal features
key_features = ['V14', 'V17', 'V4', 'V1', 'V11', 'V12']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    axes[i].hist(legit[feat].clip(-10, 10), bins=60, alpha=0.6,
                 color='#378ADD', label='Legit', density=True)
    axes[i].hist(fraud[feat].clip(-10, 10), bins=60, alpha=0.6,
                 color='#E24B4A', label='Fraud', density=True)
    axes[i].set_title(f'{feat}', fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(labelsize=8)

plt.suptitle('Key PCA feature distributions: Fraud vs Legitimate', y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('V14 shows the clearest separation — strongest fraud signal.')

## 5. Correlation Heatmap

In [ ]:
# Correlation of features with Class label
v_cols = [f'V{i}' for i in range(1, 15)]  # Top 14 for readability
corr_with_class = df[v_cols + ['Amount', 'Class']].corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E24B4A' if x > 0 else '#378ADD' for x in corr_with_class]
corr_with_class.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature correlation with fraud label (Class)', fontweight='bold')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_with_fraud.png', dpi=150, bbox_inches='tight')
plt.show()